# Convert positions into description of the mutation's context

In [1]:
import pandas as pd
import numpy as np

pd.options.display.max_columns = None

In [2]:
# Read reference genome
genome = dict()

with open('../../../datasets/genomics/reference_genome/U00096_raw.txt') as f:
    genome['U00096'] = f.read().replace('\n', '')
    
with open('../../../datasets/genomics/reference_genome/JX871397_raw.txt') as f:
    genome['JX871397'] = f.read().replace('\n', '')

# Map position ranges to their context
Coding regions, intergenic or overlaps. 
Intergenics are labeled in the order of the top strand as X/Y. 
Regions where gene overlaps are labeled as X|Y.

Use a cursor that goes through the genome.

In [3]:
# Gene map
gene_map = pd.read_csv('../../../datasets/genomics/reference_genome/gene_map.tsv', sep = '\t')

# Remove cryptic prophages
gene_map = gene_map[~gene_map.target_type.isin(['misc_feature'])]
gene_map.head()

,gene_name,chrom,start,end,strand,target_type,product,locus_tag,ecocyc,uniprot,synonyms
0,thrL,U00096,190,255,+,protein_coding,thr operon leader peptide,b0001,EG11277,P0AD86,ECK0001
1,thrA,U00096,337,2799,+,protein_coding,fused aspartate kinase/homoserine dehydrogenase 1,b0002,EG10998,P00561,"ECK0002,Hs,thrA1,thrA2,thrD"
2,thrB,U00096,2801,3733,+,protein_coding,homoserine kinase,b0003,EG10999,P00547,ECK0003
3,thrC,U00096,3734,5020,+,protein_coding,threonine synthase,b0004,EG11000,P00934,ECK0004
4,yaaX,U00096,5234,5530,+,protein_coding,DUF2502 domain-containing protein YaaX,b0005,G6081,P75616,ECK0005


In [4]:
regions = []

chroms = ['U00096', 'JX871397']

for chrom in chroms:

    # Gene map for this chromosome
    gene_map_chrom = gene_map[(gene_map.chrom == chrom)].reset_index(drop = True)
    
    # Convert position list to event list
    event_lines = []
    
    for l in range(len(gene_map_chrom)):
    
        line = gene_map_chrom.iloc[l]
        gene_name = line.gene_name
        strand = '>' if line.strand == '+' else '<'
        start = line.start
        end = line.end + 1
        
        event_lines += [[gene_name, 'start', start, strand, line.target_type]]
        event_lines += [[gene_name, 'end', end, strand, line.target_type]]
        
    events = pd.DataFrame(event_lines, columns = ['gene_name', 'event', 'pos', 'strand', 'target_type'])
    
    # Sort events
    events.sort_values(['pos', 'event'], ascending = [True, False], inplace = True)
    
    open_genes = [] # genes that are overlapping the current cursor position
    open_genes_strand = []
    open_genes_target = []

    # Chromosome
    if chrom == 'U00096':
        last_gene = 'yjtD' # last gene to be closed, for intergenic regions
        last_strand = '>'
        last_target = 'yjtD'
        last_pos = 1

    # Phage
    else:
        last_gene = 'end'
        last_strand = '>'
        last_target = 'end'
        last_pos = 1
        
    # Convert events to contexts
    for l in range(len(events)):
        
        # Is the last region intergenic?
        if not open_genes:
            
            # Open a new gene
            assert events.iloc[l].event == 'start'
            line = [chrom, last_pos, events.iloc[l].pos - 1, last_gene + '/' + events.iloc[l].gene_name, last_strand + '/' + events.iloc[l].strand, 'intergenic']
            open_genes += [events.iloc[l].gene_name]
            open_genes_strand += [events.iloc[l].strand]
            open_genes_target += [events.iloc[l].target_type]
            last_gene = events.iloc[l].gene_name
            last_strand = events.iloc[l].strand
            last_target = events.iloc[l].target_type
            
        # Are we adding a new overlap?
        elif events.iloc[l].event == 'start':
            
            # Write the genes that were already open (not including the new gene)
            line = [chrom, last_pos, events.iloc[l].pos - 1, '|'.join(open_genes), '|'.join(open_genes_strand), '|'.join(open_genes_target)]
            open_genes += [events.iloc[l].gene_name]
            open_genes_strand += [events.iloc[l].strand]
            open_genes_target += [events.iloc[l].target_type]
            last_gene = events.iloc[l].gene_name
            last_strand = events.iloc[l].strand
            last_target = events.iloc[l].target_type
            
        # Are we closing a gene?
        elif events.iloc[l].event == 'end':
            # Write the genes that were already open (current gene should be here)
            assert events.iloc[l].gene_name in open_genes
            line = [chrom, last_pos, events.iloc[l].pos - 1, '|'.join(open_genes), '|'.join(open_genes_strand), '|'.join(open_genes_target)]
            open_genes_strand.pop(open_genes.index(events.iloc[l].gene_name))
            open_genes_target.pop(open_genes.index(events.iloc[l].gene_name))
            open_genes.remove(events.iloc[l].gene_name)
            
        # Add orientation
        
        # Add to the list of regions, unless it's a zero-length region
        if last_pos != events.iloc[l].pos:
            regions += [line]
            
        last_pos = events.iloc[l].pos
        
    # Add a last line to close the circular chromosome
    if chrom == 'U00096':
        regions += [[chrom, 4641629, 4641652, 'yjtD/thrL', '>/>', 'intergenic']]

    else:
        pass
        
# Aggregate into a DF
position_to_context = pd.DataFrame(regions, columns = ['chrom', 'start', 'end', 'context', 'strand', 'target_type'])

position_to_context.head(5)

,chrom,start,end,context,strand,target_type
0,U00096,1,189,yjtD/thrL,>/>,intergenic
1,U00096,190,255,thrL,>,protein_coding
2,U00096,256,336,thrL/thrA,>/>,intergenic
3,U00096,337,2799,thrA,>,protein_coding
4,U00096,2800,2800,thrA/thrB,>/>,intergenic


In [5]:
position_to_context['context_part'] = position_to_context.groupby(['context']).apply('cumcount') + 1

In [6]:
position_to_context.to_csv('../../../datasets/genomics/04_gene_attribution/intermediate/position_to_context.tsv', sep = '\t', index = False, encoding='utf-8-sig')

# Find the context of each variant

In [7]:
# Open mutations file
mutations = pd.read_csv('../../../datasets/genomics/03_variant_filtering/results/d_filtered.tsv', sep = '\t', keep_default_na = False)

# Add variant identifier
mutations['v_string'] = mutations.chrom + '_' + mutations.pos.astype(str) + '_' + mutations.ref + '_' + mutations.alt + '_' + mutations.delta.astype(str)

mutations.head()

# next step: merge the two pos_to_contexts

,name,pop,origin,gen,freq,chrom,pos,ref,alt,wave,type,delta,pos_end,locus,rep_len,rep_nt,ad,td,edits,crosses,raw_pos,raw_freq,raw_ref,raw_alt,raw_type,read_pop,back_alt,back_delta,collapsed_rows,max_f,apparition,fixation,v_string
0,EA00,A,A,0,0.0,JX871397,4929,C,T,1,SNP,0.0,4930,CTTATTGC.C.AGTTCGGCA,2,.,0.0,NA,None,NA,NA,NA,,,,A,,NA,NA,0.2968,21,inf,JX871397_4929_C_T_0.0
1,EA01,A,A,1,0.0,JX871397,4929,C,T,1,SNP,0.0,4930,CTTATTGC.C.AGTTCGGCA,2,.,0.0,NA,None,NA,NA,NA,,,,A,,NA,NA,0.2968,21,inf,JX871397_4929_C_T_0.0
2,EA02,A,A,2,0.0,JX871397,4929,C,T,1,SNP,0.0,4930,CTTATTGC.C.AGTTCGGCA,2,.,0.0,NA,None,NA,NA,NA,,,,A,,NA,NA,0.2968,21,inf,JX871397_4929_C_T_0.0
3,EA03,A,A,3,0.0,JX871397,4929,C,T,1,SNP,0.0,4930,CTTATTGC.C.AGTTCGGCA,2,.,0.0,NA,None,NA,NA,NA,,,,A,,NA,NA,0.2968,21,inf,JX871397_4929_C_T_0.0
4,EA04,A,A,4,0.0,JX871397,4929,C,T,1,SNP,0.0,4930,CTTATTGC.C.AGTTCGGCA,2,.,0.0,NA,None,NA,NA,NA,,,,A,,NA,NA,0.2968,21,inf,JX871397_4929_C_T_0.0


# Get contexts
Add a gene_name and a strand column. Intergenic contexts are separated by "/", overlapping genes by "|".

In [8]:
# List of unique mutations, pos, ref, alt
unique_mutations = mutations.groupby(['chrom', 'pos', 'pos_end', 'ref', 'alt', 'type', 'delta', 'v_string']).agg(max_f=('freq', 'max'), pops=('pop', lambda x: ''.join(x.unique()))).reset_index()
unique_mutations.sort_values(['chrom', 'pos'], inplace = True)

In [9]:
unique_mutations_dfs = []

for chrom in chroms:

    unique_mutations_chrom = unique_mutations[unique_mutations.chrom == chrom].reset_index(drop = True)
    position_to_context_chrom = position_to_context[position_to_context.chrom == chrom].reset_index(drop = True)

    contexts, context_parts, strands, starts, ends, targets = [], [], [], [], [], []
    
    context_cursor = 0 # currently considered context
    current_context = position_to_context_chrom.iloc[context_cursor] # first available context
    
    # Attribute contexts assuming they come in order
    for i in range(len(unique_mutations_chrom)):
        
        line = unique_mutations_chrom.iloc[i]
        
        # Current mutations is out of the considered context
        while line.pos > current_context.end:
            context_cursor += 1 # Move to the next context
            current_context = position_to_context_chrom.iloc[context_cursor]
        
        contexts.append(current_context.context)
        context_parts.append(current_context.context_part)
        strands.append(current_context.strand)
        targets.append(current_context.target_type)
        starts.append(current_context.start)
        ends.append(current_context.end)
            
    unique_mutations_chrom['context'] = contexts
    unique_mutations_chrom['context_part'] = context_parts
    unique_mutations_chrom['strand'] = strands
    unique_mutations_chrom['target_type'] = targets
    unique_mutations_chrom['start'] = starts
    unique_mutations_chrom['end'] = ends

    unique_mutations_dfs.append(unique_mutations_chrom)

unique_mutations = pd.concat(unique_mutations_dfs)

In [10]:
abb_df = pd.read_csv('../../../datasets/genomics/01_variant_calling/results/abbreviations.tsv', sep = '\t')
abb_dict = {i: j for i, j in zip(abb_df.short.values, abb_df.long.values)}

In [11]:
import Bio.Seq as Seq

import fuzzysearch
import Levenshtein
import traceback

# For fuzzy search
crop = 10 # how many bases to remove on each side of the inserted sequence
max_l_dist = 5 # max levhenstein distance for a match

large_thr = 30 # bp beyond which it's considered a large change
check_inversion = 5 # bp beyond which to check if a mutation is an inversion

def get_aa_change(row, g):
    '''Get the change in aminoacid at position row.pos, with g = (name, strand). Returns (posingene, aa_pos)'''
    
    # Find the gene's WT sequence
    start = int(gene_map.loc[gene_map.gene_name == g[0]].start.values[0])
    end = int(gene_map.loc[gene_map.gene_name == g[0]].end.values[0])
    chrom = gene_map.loc[gene_map.gene_name == g[0]].chrom.values[0]
    
    # Is the gene non-coding?
    if g[2] != 'protein_coding':
        return 'noncoding', '', '', '', '', '', ''
    
    gene_strand = g[1]
    
    # Make new sequence (as it is on the top strand)
    old_seq = genome[chrom][start - 1: end]
    new_seq = genome[chrom][start - 1: row.pos - 1] + row.alt + genome[chrom][row.pos - 1 + len(row.ref): end + len(row.ref) - 1]

    # prfB is translated with a programmed +1 frameshift (ribosomal slippage, see the GFF): base 3036207 is not read
    skipped = {'prfB': 3036207}.get(g[0])
    if skipped:
        shift_new = len(row.alt) - len(row.ref) if row.pos < skipped else 0
        old_seq = old_seq[:skipped - start] + old_seq[skipped - start + 1:]
        new_seq = new_seq[:skipped - start + shift_new] + new_seq[skipped - start + shift_new + 1:]

    # Make ORF in the right direction
    if gene_strand == '>':
        old_orf = old_seq
        new_orf = new_seq
    else:
        old_orf = Seq.reverse_complement(old_seq)
        new_orf = Seq.reverse_complement(new_seq)
    
    # Find aa position
    position_in_gene = (row.pos - start) if (gene_strand == '>') else (end - row.pos) # position within gene
    if skipped and ((gene_strand == '>' and row.pos > skipped) or (gene_strand != '>' and row.pos < skipped)):
        position_in_gene -= 1 # one base upstream of the mutation is not read
    aa_pos = int(position_in_gene // 3) # zero-indexed
    
    # Get codons
    codon_ref = old_orf[aa_pos*3:aa_pos*3+3]
    codon_alt = new_orf[aa_pos*3:aa_pos*3+3]
    
    # Trim to a mutliple of 3 for translation
    if len(old_orf) % 3 != 0:
        old_orf = old_orf[:-(len(old_orf) % 3)]
    if len(new_orf) % 3 != 0:
        new_orf = new_orf[:-(len(new_orf) % 3)]
    
    # Predict the mutation's effect
    old_prot = Seq.translate(old_orf)
    new_prot = Seq.translate(new_orf)
    
    try:
        aa_ref = old_prot[aa_pos]
        
        # If it's a SNP
        if (len(row.ref) == 1) and (len(row.alt) == 1):
            
            # Get substitution
            aa_alt = new_prot[aa_pos]
            effect = aa_ref + str(aa_pos + 1) + aa_alt
            
            # Check that the codons check out
            assert Seq.translate(codon_ref) == aa_ref
            assert Seq.translate(codon_alt) == aa_alt
            
            # Get mutation category
            if aa_ref == aa_alt:
                cat = 'synonymous'
            elif aa_ref == '*':
                cat = 'extension'
            elif aa_alt == '*':
                cat = 'nonsense'
            else:
                cat = 'missense'
        
        # If it's an indel
        else:
            
            # Get shift size
            shift = len(row.alt) - len(row.ref)
            aa_alt = '::' + str(shift) if shift >= 0 else 'Δ' + str(-shift)
            effect = aa_ref + str(aa_pos + 1) + aa_alt
            
            # Get mutation category
            cat = 'aa_indel' if shift % 3 == 0 else 'frameshift'
    except:
        print(traceback.print_exc())
        display(row)
        aa_ref = 'pute'
        aa_alt = 'pute'
        effect = 'pute'
        
    return cat, effect, aa_ref, str(aa_pos + 1), aa_alt, codon_ref, codon_alt

def covered_genes(x):
    '''Find genes that overlap with a large deletion.'''
    end = x.pos + len(x.ref)
    full = gene_map[(x.pos > gene_map.start) & (end < gene_map.end)].gene_name
    partial_end = gene_map[(x.pos < gene_map.start) & (end > gene_map.start)].gene_name
    partial_start = gene_map[(end > gene_map.end) & (x.pos < gene_map.end)].gene_name
    return ', '.join(['δ' + g for g in partial_end] + ['Δ' + g for g in full] + ['δ' + g for g in partial_start if g not in list(partial_end)])

def covered_genes_ins(start, end):
    '''Find genes that overlap with a large insertion.'''
    
    # Find genes entirely within the inserted sequence
    genes = gene_map[(start < gene_map.start) & (end > gene_map.end)].gene_name

    if len(genes):
        return '-'.join(genes)

    # If no gene is inside, find the origin of the sequence
    else:
        origin = gene_map[(start > gene_map.start) & (end < gene_map.end)].gene_name
        if len(origin):
            return f'{origin.iloc[0]}({start}:{end})'
        else:
            return f'unspecified({start}:{end})'

def find_insertion_origin(seq):
    '''Search the genome for the origin of an insertion sequence, with fuzzy matching.'''
    
    matches = fuzzysearch.find_near_matches(seq[crop:-crop], genome['U00096'], max_l_dist = max_l_dist)
    
    if not matches: # if there's no hit, try again with a looser threshold
        matches = fuzzysearch.find_near_matches(seq[crop:-crop], genome['U00096'], max_l_dist = 10)
        
    if not matches: # still no matches
        return 'unspecified' # an unspecified one

    else:
        best_match = matches[np.argmin([d.dist for d in matches])]
        covered = covered_genes_ins(best_match.start, best_match.end)

        return covered
    
def get_effect(row):
    
    # Expand abbreviations    
    if 'bp' in row.ref:
        row.ref = abb_dict[row.ref]
    
    if 'bp' in row.alt:
        row.alt = abb_dict[row.alt]
        
    # Expand REPs
    if '×' in row.ref:
        row.ref = row.ref[0] * round(row.delta)
        
    if '×' in row.alt:
        row.alt = row.alt[0] * round(row.delta)
    
    # Large changes
    if len(row.ref) > check_inversion and len(row.alt) > check_inversion:
        row['aa_ref'], row['aa_pos'], row['aa_alt'] = '', '', ''
        row['codon_ref'] = row['codon_alt'] = ''
        
        # Is it an inversion?
        d = Levenshtein.distance(row.ref, Seq.reverse_complement(row.alt))
        row['cat'] = 'inversion' if d <= 2 else 'indel'
        row['effect'] = 'inversion' if d <= 2 else 'indel'
        row['codon_ref'] = row['codon_alt'] = ''
        
    elif len(row.ref) > large_thr:
        row['aa_ref'], row['aa_pos'], row['aa_alt'] = '', '', ''
        row['cat'] = 'large_deletion'
        row['effect'] = covered_genes(row)
        row['codon_ref'] = row['codon_alt'] = ''

    elif len(row.alt) > large_thr:
        row['aa_ref'], row['aa_pos'], row['aa_alt'] = '', '', ''
        row['cat'] = 'large_insertion'
        row['effect'] = '::' + find_insertion_origin(row.alt)
        row['codon_ref'] = row['codon_alt'] = ''
    
    # Small changes
    else:
        
        # Intergenic
        if '/' in row.context:
            row['aa_ref'], row['aa_pos'], row['aa_alt'] = '', '', ''
            row['cat'] = 'intergenic'
            row['effect'] = ''
            row['codon_ref'] = row['codon_alt'] = ''

        # Intragenic
        else:
            
            # Make a list of g = (name, strand)
            genes = [g for g in zip(row.context.split('|'), row.strand.split('|'), row.target_type.split('|'))] # name, strand, target_type

            effects_list = [get_aa_change(row, g) for g in genes]
            variables_list = [i for i in zip(*effects_list)] # reciprocal of zip

            row['cat'] = '|'.join(variables_list[0])
            row['effect'] = '|'.join(variables_list[1])
            row['aa_ref'] = '|'.join(variables_list[2])
            row['aa_pos'] = '|'.join(variables_list[3])
            row['aa_alt'] = '|'.join(variables_list[4])
            row['codon_ref'] = '|'.join(variables_list[5])
            row['codon_alt'] = '|'.join(variables_list[6])
            
    # Reversions
    if row['type'] == 'REV':
        row['aa_alt'] = row['aa_ref']
        row['cat'] = 'reversion'
        row['effect'] = 'reversion'
            
    return row

In [12]:
umut = unique_mutations.reset_index(drop = True)

In [13]:
from tqdm import tqdm

tqdm.pandas() # for progress bar
umut = umut.progress_apply(get_effect, axis = 1)

  0%|          | 0/7740 [00:00<?, ?it/s]

  1%|▏         | 97/7740 [00:00<00:07, 968.76it/s]

  3%|▎         | 196/7740 [00:00<00:07, 980.81it/s]

  4%|▍         | 295/7740 [00:00<00:10, 733.30it/s]

  5%|▍         | 375/7740 [00:00<00:16, 455.34it/s]

  6%|▌         | 468/7740 [00:00<00:13, 534.05it/s]

... (85 outputs left out)

 98%|█████████▊| 7562/7740 [00:12<00:00, 523.56it/s]

 98%|█████████▊| 7622/7740 [00:12<00:00, 515.52it/s]

100%|█████████▉| 7722/7740 [00:12<00:00, 625.48it/s]

100%|██████████| 7740/7740 [00:13<00:00, 589.59it/s]

# Keep only the necessary columns

In [14]:
# Remove max_Intermediate frames/ pops
# Repeat-length changes (REP) get the same ref and alt strings above, so a +-1 change was labelled 'synonymous' or 'aa_indel';
# any length change that is not a multiple of 3 inside a coding sequence shifts the frame
fs = (umut.type == 'REP') & umut.cat.isin(['synonymous', 'aa_indel']) & (umut.delta.round() % 3 != 0)
umut.loc[fs, 'cat'] = 'frameshift'
umut.loc[fs, 'effect'] = umut.loc[fs].apply(lambda r: r.aa_ref + str(r.aa_pos) + ('::' + str(int(round(r.delta))) if r.delta > 0 else 'Δ' + str(int(round(-r.delta)))), axis = 1)
print(f'{fs.sum()} repeat-length changes relabelled as frameshifts')

mutation_to_effect = umut[['v_string', 'context', 'context_part', 'strand', 'effect', 'codon_ref', 'codon_alt', 'cat', 'target_type', 'aa_ref', 'aa_pos', 'aa_alt']].reset_index(drop = True)

493 repeat-length changes relabelled as frameshifts


# Add gene product info

In [15]:
gene_name_to_product = {gene_map.iloc[i].gene_name: gene_map.iloc[i]['product'] for i in range(len(gene_map))}

In [16]:
def context_to_product(context):
    if '/' in context:
        return '/'.join([gene_name_to_product[i] for i in context.split('/')])
    else:
        return '|'.join([gene_name_to_product[i] for i in context.split('|')])

In [17]:
mutation_to_effect['product'] = mutation_to_effect['context'].apply(context_to_product)

# Merge with the history frame

In [18]:
d_eff = mutations.merge(mutation_to_effect, how = 'left')

In [19]:
d_eff.to_csv('../../../datasets/genomics/04_gene_attribution/intermediate/d_eff.tsv', sep = '\t', index = False, encoding='utf-8-sig')